# CS604 Colab launcher — Ana: LoRA r24 recipe ablation (pilot scale)

Which changes took LoRA from 0.717 to 0.785? Runs the arms in `configs/ablation_lora_r24.yaml`, each one change from `lora_r24_336_recipe.yaml`, on a 1,000-class pilot subset (40 fit images/class, 3 epochs) and scores them on a labelled 5/class holdout:

| Arm | Change |
|---|---|
| A | none (full recipe) |
| B | no norms / biases / pos_embed / cls_token |
| C | rank 16 instead of 24 |
| D | linear head from a linear probe (LP-FT) instead of the prototype cosine head |
| E | head lr 1e-4 instead of 3e-3 |
| F | B + C + D + E (≈ the 0.717 run's setup) |
| G | D with head lr 1e-4 (probe head, slow head lr) |
| H | linear head started from prototypes (instead of the cosine head) |
| I | DoRA instead of LoRA (extension: learned magnitude per output, +0.10% params) |

Both starting heads are rebuilt from the pilot's fit images only, so no head has seen the holdout. Each arm runs in its own process. If the session drops, Run all again: finished arms are skipped.

About 1 hour on an A100: data download ~15 min, clean heads ~5 min, 9 arms × ~6 min (finished arms are skipped). Results go to `CS604_RUNS/ablation_ablation_lora_r24/`.

Runtime: **A100 GPU**. Use **Runtime → Run all**.

In [1]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336_recipe"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/ablation_lora_r24.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [2]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ··········
Cloning into '/content/repo'...
remote: Enumerating objects: 1222, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1222 (delta 78), reused 74 (delta 63), pack-reused 1112 (from 1)
Receiving objects: 100% (1222/1222), 1.23 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (716/716), done.
/content/repo
ana-branch
38a713c


## 2. Install requirements

In [3]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [4]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

Mounted at /content/drive
Writing runs to /content/drive/.shortcut-targets-by-id/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR/CS604_RUNS
Existing items: ['full_ft_20261005_074553_id962394', 'full_ft_20261006_031830', 'headinit_full_ft_336_20261006_110343', 'prototypes_336.pt', 'prototypes_336.json', 'full_ft_20261006_122039', 'lora_20261007_051926', 'qq_selected_layer_4bae5d64c9c7ecfb', 'last_blocks_20261008_123344_id969549', 'qq_prefix_ablation_336_seed42_v1', 'pilot_adapter_336_20261009_125653', 'adapter_20261009_131150', 'prototypes_224.json', 'prototypes_224.pt', 'last_blocks_20261010_022115', 'val_team_Wenqi.txt', 'prefix_20261010_000404_id974631', 'qq_bitfit_ablation_336_subset_seed42_v2', 'prototypes_336.usage_metadata.json', 'bitfit_20261011_024724_id976355']


## 4. Download data to `/content/data`

In [5]:
!python scripts/download_data.py --dest {DATA_DIR}

[download] snapshot doem1997/cs604-course-data -> /content/data
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.05M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1:34:53, 7.83MB/s]
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1:35:09, 7.81MB/s]
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1:35:31, 7.78MB/s]
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1:48:39, 6.84MB/s]

Fetching 7 files:  29% 2/7 [00:00<00:00, 12.59it/s]
Reconstructing (incomplete total...):   0% 1.05M/44.6G [00:00<1:49:00, 6.82MB/s]
Reconstructing (incomplete total...):   0% 29.4M/44.6G [00:00<21:07, 35.1MB/s]  

Reconstructing (incomplete total...):  18% 7.89G/44.6G 

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [6]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

Already have /content/drive/MyDrive/CS604_RUNS/prototypes_336.pt


## 5. Ablation

In [7]:
import subprocess
import sys

proc = subprocess.run([sys.executable, "-u", "scripts/ablation.py", "--spec", SPEC,
                       "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
                       "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR])
if proc.returncode != 0:
    end_runtime(f"ablation failed (exit {proc.returncode}); see the output above")
    raise SystemExit("ablation failed")

## 6. Results

`delta_vs_A_points`: change in pilot holdout top-1 (percentage points) against the full recipe. Negative = turning that piece off hurts, so that piece helped. Treat differences under ~0.3 points as ties (one seed).

In [8]:
from pathlib import Path

import pandas as pd

table = pd.read_csv(Path(DRIVE_OUT) / "ablation_ablation_lora_r24" / "ablation.csv")
display(table[["arm", "changes", "final_val_acc", "delta_vs_A_points", "best_val_acc", "final_val_loss",
               "early_dip", "backbone_pct", "wall_min"]])

,arm,changes,final_val_acc,delta_vs_A_points,best_val_acc,final_val_loss,early_dip,backbone_pct,wall_min
0,A_full_recipe,{},0.8984,0.00,0.8984,1.7482,False,4.65,6.3
1,B_no_extras,"{""unfreeze_extra"": []}",0.8970,-0.14,0.8970,1.7534,False,4.12,5.9
2,C_rank16,"{""lora_rank"": 16, ""lora_alpha"": 16}",0.8918,-0.66,0.8918,1.7581,False,3.28,6.3
3,D_probe_head,"{""head_type"": ""linear"", ""head_init"": ""probe""}",0.8692,-2.92,0.8692,1.9072,False,4.65,6.3
4,E_head_lr_1e-4,"{""head_lr"": 0.0001}",0.8370,-6.14,0.8370,2.2641,False,4.65,6.3
5,F_all_four_off,"{""unfreeze_extra"": [], ""lora_rank"": 16, ""lora_...",0.8538,-4.46,0.8538,1.9938,False,2.74,6.0
6,G_probe_head_slow_lr,"{""head_type"": ""linear"", ""head_init"": ""probe"", ...",0.8572,-4.12,0.8588,1.9735,False,4.65,6.5
7,H_linear_prototype_head,"{""head_type"": ""linear"", ""head_init"": ""prototype""}",0.8892,-0.92,0.8892,1.7860,False,4.65,6.3
8,I_dora,"{""lora_dora"": true}",0.8996,0.12,0.8996,1.7464,False,4.75,7.7


## 7. End the runtime

In [9]:
end_runtime("all cells finished; ending the runtime")

all cells finished; ending the runtime
Drive flushed and unmounted
